# Importing Necessary Packages

In [3]:
import os
import numpy as np
import zipfile
import tarfile
import gzip
import nibabel as nib
import matplotlib.pyplot as plt
from google.colab import drive

# Standard Histogram Equalization (HE) Algorithm

In [4]:
def global_histogram_equalization(nii_image):
  # Getting the Image as a 3D Array of values
  image_arr = (nii_image.get_fdata()).astype(int)
  histogram = np.bincount(image_arr.ravel(), minlength=2**16)
  # Getting Cummulative Histogram and Intensity
  cum_histogram = np.cumsum(histogram)
  del(histogram)
  intensity = (cum_histogram * (2**16 - 1) / image_arr.size).astype(int)
  del(cum_histogram)
  image_arr = intensity[image_arr]

  return image_arr

# Local Histogram Equalization (LHE) Algorithm

In [5]:
def pad(image_arr):
  pad_sizes = ((1, 1), (1, 1), (1, 1))
  return np.pad(image_arr, pad_sizes, mode='edge')

In [6]:
# Do Histogram Equalization Locally to get the anchor pixel value
def do_he_on_region(image_region):
  histogram = np.bincount(image_region.ravel(), minlength= 256)
  cum_histogram = np.cumsum(histogram)
  intensity = (cum_histogram * (255) / image_region.size).astype(int)
  return intensity[image_region]

In [10]:
def local_histogram_equalization(nii_image):
    # Getting the Image as a 3D Array of values
    image_arr = (nii_image.get_fdata()).astype(int)
    padded_image_arr = pad(image_arr)

    # Defining output array and enhancing the image pixels locally
    enhanced_arr = np.zeros(image_arr.shape)
    indices = np.indices((3, 3, 3)).reshape(3, -1).T

    for i in range(1, padded_image_arr.shape[0] - 1):
      for j in range(1, padded_image_arr.shape[1] - 1):
        for k in range(1, padded_image_arr.shape[2] - 1):
          # Get the 3x3x3 region using advanced indexing
          region_indices = indices + np.array([i, j, k]) - 1
          region = padded_image_arr[region_indices[:, 0], region_indices[:, 1], region_indices[:, 2]]
          enhanced_arr[i - 1, j - 1, k - 1] = do_he_on_region(region)[13]

    return enhanced_arr